# Cosmos SAE Training and Feature Browser

Run this notebook on the RunPod checkout. It assumes the repo is at `/workspace/cosmos-sae-reasoner` and that the Cosmos3-Nano Hugging Face cache is under `/workspace/.cache/huggingface`.

The default cells reuse the small activation smoke set. To run a larger experiment, change `ACTIVATION_DIR`, `MAX_EXAMPLES`, `TRAIN_STEPS`, and output names in the config cell.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time
import sys


ROOT = Path('/workspace/cosmos-sae-reasoner')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODEL_ID = 'nvidia/Cosmos3-Nano'
LAYER = 18

# Combined PhysicalAI dataset. Change DATASET_SAMPLE_COUNT before running the manifest-build cell.
DATASET_SAMPLE_COUNT = 100
VANTAGE_FRACTION = 0.5  # image share; remainder is driving video
DATASET_SEED = 0
MANIFEST = ROOT / f'outputs/sae_reasoner/manifests/physicalai_combined_{DATASET_SAMPLE_COUNT}.jsonl'
ACTIVATION_DIR = ROOT / f'outputs/sae_reasoner/activations/physicalai_combined_l{LAYER}_{DATASET_SAMPLE_COUNT}'
SAE_OUT = ROOT / f'outputs/sae_reasoner/saes/physicalai_combined_l{LAYER}_{DATASET_SAMPLE_COUNT}.pt'
FEATURES_OUT = ROOT / f'outputs/sae_reasoner/reports/physicalai_combined_l{LAYER}_{DATASET_SAMPLE_COUNT}_features.jsonl'
REPORT_OUT = ROOT / f'outputs/sae_reasoner/reports/physicalai_combined_l{LAYER}_{DATASET_SAMPLE_COUNT}_features.html'
NEIGHBORS_OUT = ROOT / f'outputs/sae_reasoner/reports/physicalai_combined_l{LAYER}_{DATASET_SAMPLE_COUNT}_neighbors.jsonl'
NEIGHBOR_REPORT_OUT = ROOT / f'outputs/sae_reasoner/reports/physicalai_combined_l{LAYER}_{DATASET_SAMPLE_COUNT}_neighbors.html'

MAX_EXAMPLES = DATASET_SAMPLE_COUNT
TRAIN_STEPS = 500
BATCH_SIZE = 1024
EXPANSION_FACTOR = 16
TOP_K = 32
RECON_LOSS = 'mse'
FEATURE_L1_COEFF = 0.0
FEATURE_IDS = ''  # empty means first 128 features
TOP_N = 20

os.environ['HF_HOME'] = '/workspace/.cache/huggingface'
token_path = Path('/root/.cache/huggingface/token')
if token_path.exists():
    token = token_path.read_text(encoding='utf-8').strip()
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token

def run(cmd: str) -> None:
    print(f'$ {cmd}', flush=True)
    start = time.time()
    subprocess.run(cmd, cwd=ROOT, shell=True, check=True)
    print(f'done in {time.time() - start:.1f}s', flush=True)

print('repo', ROOT)
print('activation_dir', ACTIVATION_DIR)
print('sae_out', SAE_OUT)
print('features_out', FEATURES_OUT)
print('report_out', REPORT_OUT)
print('neighbors_out', NEIGHBORS_OUT)
print('neighbor_report_out', NEIGHBOR_REPORT_OUT)

## Build Combined Manifest

Build a configurable mixed image/video manifest from the first-party PhysicalAI recipes. This only writes metadata; media stays remote until activation collection.

In [ ]:
# Build a combined PhysicalAI manifest with a selectable number of samples.
import random

def build_combined_physicalai_manifest(total: int = DATASET_SAMPLE_COUNT, *, vantage_fraction: float = VANTAGE_FRACTION, seed: int = DATASET_SEED) -> None:
    image_count = int(round(total * vantage_fraction))
    video_count = max(0, total - image_count)
    tmp_dir = ROOT / 'outputs/sae_reasoner/manifests/_parts'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    vantage = tmp_dir / f'physicalai_vantage_{image_count}.jsonl'
    driving = tmp_dir / f'physicalai_driving_{video_count}.jsonl'
    if image_count:
        run(f'python -m tools.sae_reasoner build-corpus-manifest --source recipe --recipe physicalai-vantage --max-records {image_count} --seed {seed} --output {vantage}')
    if video_count:
        run(f'python -m tools.sae_reasoner build-corpus-manifest --source recipe --recipe physicalai-driving --max-records {video_count} --seed {seed + 1} --output {driving}')
    rows = []
    for part in [vantage, driving]:
        if part.exists():
            rows.extend(json.loads(line) for line in part.read_text(encoding='utf-8').splitlines())
    random.Random(seed).shuffle(rows)
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST.write_text(''.join(json.dumps(row, ensure_ascii=True, sort_keys=True) + '\n' for row in rows), encoding='utf-8')
    print({'manifest': str(MANIFEST), 'records': len(rows), 'images_requested': image_count, 'videos_requested': video_count})

BUILD_MANIFEST = True
if BUILD_MANIFEST or not MANIFEST.exists():
    build_combined_physicalai_manifest()
else:
    print('Using existing manifest', MANIFEST)


## Manifest Browser

Use this before collecting activations to inspect prompts, tags, images, and sampled video frames directly in the notebook. This is intentionally separate from the JSONL file so multimodal rows are easy to scan.

In [ ]:
from html import escape
from IPython.display import HTML, display
import base64
import io
from PIL import Image
from tools.sae_reasoner.manifest import load_manifest
from tools.sae_reasoner.runtime.cosmos_hf import load_video_frames

def _image_data_uri(path: Path, *, max_size=(420, 320)) -> str:
    img = Image.open(path).convert('RGB')
    img.thumbnail(max_size)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

def _video_contact_sheet_data_uri(path: Path, *, frames=8, thumb=(180, 140), cols=4) -> str:
    arr = load_video_frames(str(path), max_frames=frames)
    thumbs = []
    for i, frame in enumerate(arr):
        img = Image.fromarray(frame)
        img.thumbnail(thumb)
        canvas = Image.new('RGB', (thumb[0], thumb[1] + 22), 'white')
        canvas.paste(img, ((thumb[0] - img.width) // 2, 0))
        from PIL import ImageDraw
        ImageDraw.Draw(canvas).text((6, thumb[1] + 4), f'frame {i}', fill=(0, 0, 0))
        thumbs.append(canvas)
    rows = (len(thumbs) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * thumb[0], rows * (thumb[1] + 22)), 'white')
    for i, img in enumerate(thumbs):
        sheet.paste(img, ((i % cols) * thumb[0], (i // cols) * (thumb[1] + 22)))
    buf = io.BytesIO()
    sheet.save(buf, format='JPEG', quality=85)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

def _activation_meta_by_id(activation_dir: Path) -> dict:
    path = activation_dir / 'metadata.jsonl'
    if not path.exists():
        return {}
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    return {row.get('id'): row for row in rows}

def _jupyter_file_url(path: Path) -> str:
    try:
        rel = path.resolve().relative_to(ROOT.resolve())
        return '/files/' + '/'.join(escape(part) for part in rel.parts)
    except ValueError:
        return ''

def render_manifest_browser(manifest: Path, *, activation_dir: Path | None = None, max_height: int = 760) -> None:
    records = load_manifest(manifest)
    meta_by_id = _activation_meta_by_id(activation_dir) if activation_dir else {}
    cards = []
    for record in records:
        media_html = '<div class="missing">text only</div>'
        media_path = Path(record.media_path) if record.media_path else None
        if media_path and not media_path.is_absolute():
            media_path = ROOT / media_path
        try:
            if record.media_type == 'image' and media_path:
                media_html = f'<img src="{_image_data_uri(media_path)}" />'
            elif record.media_type == 'video' and media_path:
                video_url = _jupyter_file_url(media_path)
                player = (
                    f'<video controls preload="metadata" src="{video_url}"></video>'
                    if video_url else '<div class="missing">video is outside Jupyter root</div>'
                )
                media_html = (
                    player
                    + f'<details open><summary>sampled frames</summary><img src="{_video_contact_sheet_data_uri(media_path)}" /></details>'
                    + f'<div class="path">{escape(str(media_path))}</div>'
                )
        except Exception as exc:
            media_html = f'<div class="missing">preview error: {escape(type(exc).__name__)}: {escape(str(exc))}</div>'
        meta = meta_by_id.get(record.id, {})
        chips = ''.join(f'<span>{escape(tag)}</span>' for tag in record.tags)
        token_bits = ''
        if meta:
            token_bits = (
                f'<div class="tokens"><b>{meta.get("num_tokens", "?")}</b> tokens '
                f'<code>{escape(json.dumps(meta.get("token_kind_counts", {}), sort_keys=True))}</code></div>'
            )
        cards.append(f'''
        <article class="manifest-card">
          <div class="media">{media_html}</div>
          <div class="body">
            <div class="head"><b>{escape(record.id)}</b><span>{escape(record.media_type)}</span></div>
            <p>{escape(record.prompt)}</p>
            <div class="chips">{chips}</div>
            {token_bits}
          </div>
        </article>
        ''')
    display(HTML(f'''
    <style>
      .manifest-scroll {{ max-height: {int(max_height)}px; overflow-y: auto; padding-right: 8px; border: 1px solid #e5e5e5; border-radius: 8px; padding: 10px; background: #fafafa; }}
      .manifest-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(560px, 1fr)); gap: 14px; }}
      .manifest-card {{ display: grid; grid-template-columns: 260px minmax(0, 1fr); gap: 14px; border: 1px solid #ddd; border-radius: 8px; padding: 12px; background: #fff; }}
      .manifest-card img, .manifest-card video {{ max-width: 100%; border-radius: 6px; border: 1px solid #ddd; background: #111; }}
      .manifest-card video {{ width: 100%; display: block; margin-bottom: 8px; }}
      .manifest-card details {{ margin-top: 8px; }}
      .manifest-card summary {{ cursor: pointer; color: #555; font-size: 12px; margin-bottom: 6px; }}
      .manifest-card .head {{ display: flex; justify-content: space-between; gap: 8px; font-size: 15px; }}
      .manifest-card p {{ margin: 10px 0; line-height: 1.35; }}
      .manifest-card .chips {{ display: flex; flex-wrap: wrap; gap: 6px; }}
      .manifest-card .chips span {{ border: 1px solid #ccc; border-radius: 999px; padding: 2px 7px; font-size: 12px; }}
      .manifest-card .tokens, .manifest-card .path, .manifest-card .missing {{ margin-top: 8px; color: #666; font-size: 12px; overflow-wrap: anywhere; }}
      .manifest-card code {{ font-size: 11px; }}
    </style>
    <div class="manifest-scroll"><div class="manifest-grid">{''.join(cards)}</div></div>
    '''))

render_manifest_browser(MANIFEST, activation_dir=ACTIVATION_DIR)

In [ ]:
# Quick environment check.
run('git rev-parse --short HEAD')
run('python -m tools.sae_reasoner inspect-model --model-id nvidia/Cosmos3-Nano --init-mode meta')

In [ ]:
# Optional: collect activations. Leave RUN_COLLECTION=False if you already have ACTIVATION_DIR.
RUN_COLLECTION = False

if RUN_COLLECTION:
    run(
        'python -m tools.sae_reasoner collect-activations '
        f'--model-id {MODEL_ID} '
        f'--manifest {MANIFEST} '
        f'--layer {LAYER} '
        f'--output-dir {ACTIVATION_DIR} '
        f'--max-examples {MAX_EXAMPLES}'
    )
else:
    print('Skipping collection. Using existing activation directory.')

print('activation shards:', sorted(p.name for p in ACTIVATION_DIR.glob('*.pt'))[:10])
print('metadata exists:', (ACTIVATION_DIR / 'metadata.jsonl').exists())

In [ ]:
# Inspect activation metadata and token-map coverage.
metadata_path = ACTIVATION_DIR / 'metadata.jsonl'
rows = [json.loads(line) for line in metadata_path.read_text(encoding='utf-8').splitlines()] if metadata_path.exists() else []
for row in rows[:10]:
    print({
        'id': row.get('id'),
        'media_type': row.get('media_type'),
        'num_tokens': row.get('num_tokens'),
        'token_kind_counts': row.get('token_kind_counts'),
        'visual_grid': row.get('visual_grid'),
    })

## Nearest Activation Neighbors

Use this after collecting activations to inspect local neighborhoods in activation space before SAE training.

In [ ]:
# Raw activation nearest-neighbor browser. Run after activation collection.
NEIGHBOR_MAX_TOKENS = 5000
NEIGHBOR_NUM_QUERIES = 40
NEIGHBORS_PER_QUERY = 8
NEIGHBOR_QUERY_KINDS = 'image,video'  # use '' for all, or e.g. 'text'

run(
    'python -m tools.sae_reasoner find-neighbors '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--output {NEIGHBORS_OUT} '
    f'--max-tokens {NEIGHBOR_MAX_TOKENS} '
    f'--num-queries {NEIGHBOR_NUM_QUERIES} '
    f'--neighbors {NEIGHBORS_PER_QUERY} '
    f'--query-kinds "{NEIGHBOR_QUERY_KINDS}" '
    f'--seed {DATASET_SEED}'
)
run(
    'python -m tools.sae_reasoner render-neighbor-report '
    f'--neighbors {NEIGHBORS_OUT} '
    f'--output {NEIGHBOR_REPORT_OUT}'
)
print('neighbor report:', NEIGHBOR_REPORT_OUT)


In [ ]:
# Preview neighbor rows with token metadata.
neighbor_rows = [json.loads(line) for line in NEIGHBORS_OUT.read_text(encoding='utf-8').splitlines()]
print('num query rows:', len(neighbor_rows))
for row in neighbor_rows[:5]:
    q = row['query']
    qtok = q.get('token_info') or {}
    print('QUERY', {'record_id': q.get('record_id'), 'token_index': q.get('token_index'), 'kind': qtok.get('kind'), 'visual_position': qtok.get('visual_position')})
    for neighbor in row['neighbors'][:3]:
        ntok = neighbor.get('token_info') or {}
        print('  ', {'sim': round(float(neighbor.get('similarity', 0.0)), 4), 'record_id': neighbor.get('record_id'), 'token_index': neighbor.get('token_index'), 'kind': ntok.get('kind'), 'visual_position': ntok.get('visual_position')})


In [ ]:
# Open the nearest-neighbor report inside Jupyter.
from IPython.display import IFrame, display
display(IFrame(src=str(NEIGHBOR_REPORT_OUT.relative_to(ROOT)), width='100%', height=900))


In [ ]:
# Train the SAE.
run(
    'python -m tools.sae_reasoner train-sae '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--output {SAE_OUT} '
    f'--expansion-factor {EXPANSION_FACTOR} '
    f'--top-k {TOP_K} '
    f'--recon-loss {RECON_LOSS} '
    f'--feature-l1-coeff {FEATURE_L1_COEFF} '
    f'--steps {TRAIN_STEPS} '
    f'--batch-size {BATCH_SIZE}'
)

In [ ]:
# Inspect training metrics.
metrics_path = SAE_OUT.with_suffix('.metrics.jsonl')
metrics = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines()]
print('num metric rows:', len(metrics))
for row in metrics[-10:]:
    print(row)

In [ ]:
# Find top activating examples and render the feature browser.
run(
    'python -m tools.sae_reasoner find-features '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--sae {SAE_OUT} '
    f'--feature-ids "{FEATURE_IDS}" '
    f'--top-n {TOP_N} '
    f'--output {FEATURES_OUT}'
)
run(
    'python -m tools.sae_reasoner render-feature-report '
    f'--features {FEATURES_OUT} '
    f'--output {REPORT_OUT}'
)
print('report:', REPORT_OUT)

In [ ]:
# Preview feature rows with token metadata.
feature_rows = [json.loads(line) for line in FEATURES_OUT.read_text(encoding='utf-8').splitlines()]
print('num feature rows:', len(feature_rows))
for row in feature_rows[:20]:
    token = row.get('token_info') or {}
    print({
        'feature_id': row.get('feature_id'),
        'activation': round(float(row.get('activation', 0.0)), 4),
        'record_id': row.get('record_id'),
        'token_index': row.get('token_index'),
        'kind': token.get('kind'),
        'token_text': token.get('token_text'),
        'visual_position': token.get('visual_position'),
    })

In [ ]:
# Open the rendered report inside Jupyter.
from IPython.display import IFrame, display

relative_report = REPORT_OUT.relative_to(ROOT)
display(IFrame(src=str(relative_report), width='100%', height=900))